# EfficientNet-B0 Replacement Release Candidate (Colab)

Train one new seed-44 candidate from the frozen PlantVillage splits and export its checkpoint plus provenance evidence. This is not a rerun of the historical 12-model benchmark.


## Run instructions

1. Start a fresh Colab runtime with a T4 GPU.
2. Confirm `MyDrive/PlantVillage/Datasets.zip` exists.
3. Run all cells once, in order.
4. Do not rerun the candidate-definition cell after training starts because it creates a new timestamped output identifier.
5. Keep the exported ZIP in private artifact storage; do not commit model weights to Git.


In [ ]:
# Colab setup + repo bootstrap
import os
import shutil
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

DATASET_ZIP = Path('/content/drive/MyDrive/PlantVillage/Datasets.zip')
DATASET_DIR = Path('/content/Datasets')
FORCE_REFRESH_DATASET = True  # Set False to keep existing /content/Datasets

if not DATASET_ZIP.exists():
    raise FileNotFoundError(f'Dataset zip not found: {DATASET_ZIP}')

if FORCE_REFRESH_DATASET and DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

# Use -o to overwrite stale files and avoid partial/corrupt leftovers from previous sessions.
!unzip -q -o "{DATASET_ZIP}" -d "/content/"

if not os.path.isdir('/content/Plant-Disease-Detection-CV'):
    !git clone https://github.com/WilliamKyaww/Plant-Disease-Detection-CV.git
else:
    !cd /content/Plant-Disease-Detection-CV && git pull

!pip install -q -r /content/Plant-Disease-Detection-CV/requirements.txt

In [ ]:
# Canonical repo root
from pathlib import Path
import os, sys

repo_root = Path('/content/Plant-Disease-Detection-CV')  # cloned repo path
if not (repo_root / 'src').is_dir():
    raise FileNotFoundError(f"Repo not found at {repo_root}. Run clone cell first.")

os.chdir(repo_root)
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print('Using repo root:', repo_root)
print('Executed in Colab:', os.path.isdir('/content'))

In [ ]:
# Define and validate the isolated replacement release candidate
from datetime import datetime, timezone
from pathlib import Path
import subprocess
import sys
import torch

EXPECTED_SOURCE_COMMIT = "3dd00f7a92fed4537d53a24f0764ade26e3d5946"

source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo_root,
    text=True,
).strip()

worktree_status = subprocess.check_output(
    ["git", "status", "--porcelain"],
    cwd=repo_root,
    text=True,
).strip()

if source_commit != EXPECTED_SOURCE_COMMIT:
    raise RuntimeError(
        f"Unexpected source commit: {source_commit}. "
        f"Expected: {EXPECTED_SOURCE_COMMIT}"
    )

if worktree_status:
    raise RuntimeError(
        "Research repository is not clean before training:\n"
        + worktree_status
    )

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is unavailable. Select a T4 GPU runtime.")

CANDIDATE_ID = (
    "efficientnet_b0_seed44_replacement_"
    + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
)
CANDIDATE_OUT = Path("results/release_candidates") / CANDIDATE_ID

print("Candidate:", CANDIDATE_ID)
print("Output:", CANDIDATE_OUT)
print("GPU:", torch.cuda.get_device_name(0))
print("Source commit:", source_commit)
print("Research worktree: clean")

In [ ]:
# Verify the refreshed dataset before training
from src.integrity_report import run_and_save_report

integrity = run_and_save_report(
    run_near_duplicates=False,
    out_dir=str(CANDIDATE_OUT / "integrity"),
    verbose=False,
)

if not integrity["passed"]:
    raise RuntimeError("Dataset integrity failed. Do not begin training.")

print("Dataset integrity passed.")


In [ ]:
# Validate the exact candidate configuration without training
COMMON_ARGS = [
    "--models", "efficientnet_b0",
    "--seeds", "44",
    "--epochs", "30",
    "--batch-size", "32",
    "--lr-cnn", "3e-4",
    "--weight-decay", "1e-4",
    "--num-workers", "2",
    "--scheduler", "cosine",
    "--patience", "7",
    "--class-weighting", "inverse_frequency",
    "--pretrained",
    "--amp",
    "--out-dir", str(CANDIDATE_OUT),
]

subprocess.run(
    [sys.executable, "-m", "src.run_phase2_benchmark", *COMMON_ARGS, "--dry-run"],
    check=True,
)

In [ ]:
# Train and evaluate one new EfficientNet-B0 release candidate
subprocess.run(
    [sys.executable, "-m", "src.run_phase2_benchmark", *COMMON_ARGS],
    check=True,
)

In [ ]:
# Validate, package, persist, and download the replacement candidate
from pathlib import Path
from datetime import datetime, timezone
from google.colab import files
import hashlib
import json
import shutil
import subprocess
import sys

repo = Path("/content/Plant-Disease-Detection-CV")
candidate_results = repo / CANDIDATE_OUT
checkpoint = repo / "models/phase2/efficientnet_b0/seed_44/best.pth"
run_dir = candidate_results / "runs/efficientnet_b0/seed_44"
metrics_file = run_dir / "metrics.json"

for required in (checkpoint, metrics_file):
    if not required.exists():
        raise FileNotFoundError(f"Missing required artifact: {required}")


def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()


# Validate the checkpoint in a clean subprocess to avoid Colab's
# in-memory NumPy binary incompatibility after dependency installation.
validation_script = """
from pathlib import Path
import sys
import torch

from src.model_registry import build_model

checkpoint = Path(sys.argv[1])

state_dict = torch.load(
    checkpoint,
    map_location="cpu",
    weights_only=True,
)

model = build_model(
    "efficientnet_b0",
    num_classes=15,
    pretrained=False,
)
model.load_state_dict(state_dict, strict=True)
model.eval()

with torch.inference_mode():
    output = model(torch.zeros(1, 3, 224, 224))

if tuple(output.shape) != (1, 15):
    raise RuntimeError(
        f"Unexpected model output shape: {tuple(output.shape)}"
    )

print("Checkpoint strict-load validation passed.")
print("Output shape:", tuple(output.shape))
"""

subprocess.run(
    [
        sys.executable,
        "-c",
        validation_script,
        str(checkpoint),
    ],
    cwd=repo,
    check=True,
)

metrics = json.loads(metrics_file.read_text(encoding="utf-8"))

source_commit = subprocess.check_output(
    ["git", "rev-parse", "HEAD"],
    cwd=repo,
    text=True,
).strip()

package = Path("/content/release_candidate_exports") / CANDIDATE_ID
if package.exists():
    shutil.rmtree(package)

(package / "model").mkdir(parents=True)
(package / "frozen_splits").mkdir(parents=True)

shutil.copy2(checkpoint, package / "model/best.pth")
shutil.copytree(candidate_results, package / "evidence")

shutil.copy2(
    repo / "results/split_manifests/latest_split_manifest.json",
    package / "frozen_splits/latest_split_manifest.json",
)

for name in ("train", "val", "test"):
    shutil.copy2(
        repo / f"CSV/plantvillage_{name}.csv",
        package / f"frozen_splits/plantvillage_{name}.csv",
    )

shutil.copy2(
    repo / "requirements.txt",
    package / "requirements.txt",
)

(package / "pip_freeze.txt").write_text(
    subprocess.check_output(
        [sys.executable, "-m", "pip", "freeze"],
        text=True,
    ),
    encoding="utf-8",
)

candidate_manifest = {
    "schema_version": 1,
    "candidate_id": CANDIDATE_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "status": "replacement_release_candidate",
    "source_commit": source_commit,
    "model": "efficientnet_b0",
    "seed": 44,
    "num_classes": 15,
    "checkpoint": {
        "path": "model/best.pth",
        "sha256": sha256(checkpoint),
        "size_bytes": checkpoint.stat().st_size,
    },
    "evaluation": {
        "test_accuracy": metrics["test_accuracy"],
        "test_f1_macro": metrics["test_f1_macro"],
        "best_validation_accuracy": max(
            metrics["history"]["val_acc"]
        ),
        "epochs_completed": len(
            metrics["history"]["val_acc"]
        ),
    },
    "training": {
        "epochs_requested": metrics["epochs"],
        "batch_size": metrics["batch_size"],
        "learning_rate": metrics["learning_rate"],
        "weight_decay": metrics["weight_decay"],
        "scheduler": metrics["scheduler"],
        "patience": metrics["patience"],
        "class_weighting": metrics["class_weighting"],
        "pretrained": metrics["pretrained"],
        "amp_active": metrics["amp_active"],
    },
    "provenance_note": (
        "New seed-44 replacement candidate. It is not the missing "
        "historical seed-41 checkpoint and must not inherit that "
        "checkpoint's metrics or hash."
    ),
}

(package / "candidate_manifest.json").write_text(
    json.dumps(candidate_manifest, indent=2),
    encoding="utf-8",
)

zip_path = Path(
    shutil.make_archive(
        str(package),
        "zip",
        root_dir=package.parent,
        base_dir=package.name,
    )
)

drive_dir = Path(
    "/content/drive/MyDrive/PlantVillage/model_release_candidates"
)
drive_dir.mkdir(parents=True, exist_ok=True)

drive_zip = drive_dir / zip_path.name
shutil.copy2(zip_path, drive_zip)

print(json.dumps(candidate_manifest, indent=2))
print("Archive:", zip_path)
print("Archive SHA-256:", sha256(zip_path))
print("Drive backup:", drive_zip)

files.download(str(zip_path))